# 🎓 Curso de Big Data e IA
## Bloque 7: Proyecto integrador final

**Duración estimada:** 3 horas · **Caso práctico:** Real Sporting de Gijón, La Liga 2015/16

**Datos:** [StatsBomb Open Data](https://github.com/statsbomb/open-data). Si publicas o compartes análisis basados en estos datos, indica StatsBomb como fuente y usa su logo (disponible en su *media pack*).

**Pregunta de negocio:** *¿Qué explica los puntos que sumó el Sporting? ¿Y qué jugadores rematan mejor o peor de lo esperado?*

```
DATOS CRUDOS → LIMPIEZA (Big Data) → AGREGACIÓN → MODELO DE IA → VISUALIZACIÓN → CONCLUSIONES
```

Cada alumno/a completa su propio mini proyecto y lo presenta (10-15 min).

**Antes de empezar:** sube a Colab `sporting_partidos.csv` y `sporting_eventos.csv` (Notebook 1).

## Paso 1 — Carga de los datos limpios (Bloques 1-2)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

EQUIPO = "Sporting Gijón"

try:
    partidos = pd.read_csv("sporting_partidos.csv", parse_dates=["fecha"])
    eventos = pd.read_csv("sporting_eventos.csv")
except FileNotFoundError:
    raise SystemExit("Sube sporting_partidos.csv y sporting_eventos.csv (Notebook 1) al panel de archivos de Colab.")

print(f"{len(partidos)} partidos · {len(eventos):,} eventos")

## Paso 2 — Agregación: una fila por partido (Bloque 3)

> 💡 Con datos realmente masivos, este paso se haría con PySpark en Databricks, como en el Notebook 2. Aquí usamos pandas por simplicidad; la lógica es la misma.

In [ ]:
tiros = eventos[eventos["type"] == "Shot"].copy()
tiros["gol"] = (tiros["shot_outcome"] == "Goal").astype(int)
pases = eventos[eventos["type"] == "Pass"].copy()
pases["completo"] = (pases["pass_outcome"] == "Completo").astype(int)

xg_f = tiros[tiros["team"] == EQUIPO].groupby("match_id")["shot_xg"].sum().rename("xg_favor")
xg_c = tiros[tiros["team"] != EQUIPO].groupby("match_id")["shot_xg"].sum().rename("xg_contra")
n_f = tiros[tiros["team"] == EQUIPO].groupby("match_id").size().rename("tiros_favor")
pct_pase = pases[pases["team"] == EQUIPO].groupby("match_id")["completo"].mean().mul(100).rename("pases_ok_%")

tabla = (partidos.set_index("match_id")
         .join([xg_f, xg_c, n_f, pct_pase])
         .fillna({"xg_favor": 0, "xg_contra": 0, "tiros_favor": 0}))
tabla["dif_xg"] = tabla["xg_favor"] - tabla["xg_contra"]

tabla[["jornada", "rival", "resultado", "puntos", "xg_favor", "xg_contra", "dif_xg", "pases_ok_%"]].round(2).head(10)

## Paso 3 — ¿Qué se relaciona con los puntos? (Análisis)

In [ ]:
cols = ["puntos", "goles_favor", "goles_contra", "xg_favor", "xg_contra", "dif_xg", "tiros_favor", "pases_ok_%"]
print(tabla[cols].corr()["puntos"].drop("puntos").round(2).sort_values(ascending=False))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
sns.regplot(data=tabla, x="dif_xg", y="puntos", ax=axs[0], scatter_kws={"alpha": 0.7})
axs[0].set_title("Diferencia de xG frente a puntos")
sns.regplot(data=tabla, x="pases_ok_%", y="puntos", ax=axs[1], scatter_kws={"alpha": 0.7})
axs[1].set_title("% pases completos frente a puntos")
plt.tight_layout()
plt.show()

Con solo 38 partidos las correlaciones son orientativas, no concluyentes. Y recuerda: **correlación no es causalidad**.

## Paso 4 — Modelo de IA: ¿cuánto se esperaba de cada tirador? (Bloques 5-6)

Entrenamos un Random Forest de probabilidad de gol (como en el Notebook 4). Para predecir **todos** los tiros sin que el modelo vea su
propia respuesta, usamos `cross_val_predict`: cada tiro se predice con un modelo que **no lo ha visto al entrenar**.

In [ ]:
dx = 120 - tiros["x"]
dy = tiros["y"] - 40
base = pd.DataFrame({
    "distancia": np.sqrt(dx**2 + dy**2),
    "angulo": np.arctan2(8 * dx, dx**2 + dy**2 - 16),
    "bajo_presion": tiros["under_pressure"].astype(int),
    "de_primeras": tiros["shot_first_time"].fillna(False).astype(int),
    "shot_body_part": tiros["shot_body_part"],
    "shot_type": tiros["shot_type"],
})
X = pd.get_dummies(base, columns=["shot_body_part", "shot_type"], drop_first=True)
y = tiros["gol"]

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tiros["xg_modelo"] = cross_val_predict(rf, X, y, cv=cv, method="predict_proba")[:, 1]

print(f"AUC del modelo: {roc_auc_score(y, tiros['xg_modelo']):.3f}")
print(f"AUC xG StatsBomb: {roc_auc_score(y, tiros['shot_xg']):.3f}")
print(f"Goles reales de los tiros del Sporting: {tiros.loc[tiros['team'] == EQUIPO, 'gol'].sum()}")
print(f"Goles esperados (modelo) del Sporting:  {tiros.loc[tiros['team'] == EQUIPO, 'xg_modelo'].sum():.1f}")
print(f"Goles esperados (StatsBomb) del Sporting: {tiros.loc[tiros['team'] == EQUIPO, 'shot_xg'].sum():.1f}")

## Paso 5 — Visualización de resultados (Bloque 4)

In [ ]:
jug = (tiros[tiros["team"] == EQUIPO].groupby("player")
       .agg(tiros=("gol", "size"), goles=("gol", "sum"),
            xg_modelo=("xg_modelo", "sum"), xg_statsbomb=("shot_xg", "sum"))
       .query("tiros >= 15").round(1))
jug["goles_vs_xg"] = (jug["goles"] - jug["xg_statsbomb"]).round(1)
jug = jug.sort_values("goles_vs_xg")

colores = ["#D1495B" if v < 0 else "#2E8B57" for v in jug["goles_vs_xg"]]
jug["goles_vs_xg"].plot(kind="barh", color=colores, figsize=(9, 4.5))
plt.axvline(0, color="black", lw=0.8)
plt.title("Goles menos xG por jugador (más de 15 tiros)\nVerde: marca más de lo esperado · Rojo: marca menos")
plt.xlabel("Goles - xG")
plt.ylabel("")
plt.tight_layout()
plt.show()
jug

## Paso 6 — Conclusiones (plantilla a rellenar por el alumno/a)

Completa esta sección antes de presentar:

1. **¿Qué variable se relaciona más con los puntos del Sporting?** _(tu respuesta)_
2. **¿Qué jugadores rematan por encima o por debajo de lo esperado? ¿Es una conclusión fiable con tan pocos tiros?** _(tu respuesta)_
3. **¿Qué recomendación daría a un cuerpo técnico a partir de estos datos?** _(tu respuesta)_
4. **¿Qué limitaciones tiene el análisis?** (una temporada, pocos goles, sin posición de defensores, calidad del dato...) _(tu respuesta)_
5. **Si lo llevaras a producción, ¿qué cambiarías?** (Spark en un cluster, todas las competiciones del repositorio, un dashboard en Power BI, un modelo de xG con más información...) _(tu respuesta)_

## 🎤 Guía para la presentación (10-15 min por grupo)

1. Pregunta de negocio (1 min).
2. Qué se hizo en cada etapa: datos → limpieza → agregación → modelo → visualización (5 min).
3. Resultados y hallazgos (5 min).
4. Recomendación y limitaciones (3 min).
5. Preguntas.

Recuerda citar a **StatsBomb** como fuente de los datos.